# Demographics Break Down

In [1]:
import os
os.chdir("../")

from argparse import Namespace

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from data import *
from run import *

data_path = "../excel_data"

## Get Valid Samples

Rebuild the datasets as done in the code to get an accurate representation of used samples

In [5]:
model="gemma"

args = Namespace(
    seed=0, 
    data_path=f"../{model}_multimodal_bins",
    test_path=f"../{model}_multimodal_bins_rw",
    clinical=True,
    path_lang=True,
    rad_lang=True,
    label_col="survival_days",
    sparse=False,
    survival_years=2
)

rih, rw = get_datasets(args, keys=["slide_ids", "survival_days", "survival_right_censor", "stage"]) #
rih_loader, rw_loader = DataLoader(rih), DataLoader(rw)
#dict_keys(['label', 'label_mask', 'path_lang_mask', 'path_lang', 'rad_lang_mask', 'rad_lang', 'clinical_mask', 'clinical', 'slide_ids', 'survival_days', 'survival_right_censor'])
rih_df, rw_df = pd.read_excel(f"{data_path}/PDAC_IHC.xlsx"), pd.read_excel(f"{data_path}/Combined_Clinical_Lab.xlsx")

In [ ]:
from torch.utils.data import ConcatDataset
from sklearn.model_selection import train_test_split, KFold
from lifelines import KaplanMeierFitter
import matplotlib.pyplot as plt
from sksurv.metrics import concordance_index_censored, cumulative_dynamic_auc
from torchsurv.metrics.cindex import ConcordanceIndex

all_ds = ConcatDataset([rih, rw])
all_loader = DataLoader(all_ds)
stages, time, event = [], [], []
for s in all_loader:
    stages.append(s['stage'].item())
    time.append(s['survival_days'].item())
    event.append(not s['survival_right_censor'].item())

df = pd.DataFrame({"Stage": stages, "Survival Time": time, "Survival Event": event})

tv, test_inds = train_test_split(
    np.arange(len(all_ds)),
    test_size=0.15,
    random_state=args.seed,
)

test = df.iloc[test_inds]

kf = KFold(n_splits=5, shuffle=True, random_state=args.seed)
splits = kf.split(X=tv)

for split, (train_inds, val_inds) in enumerate(splits):
    train, val = df.iloc[train_inds], df.iloc[val_inds]
    # Initialize the fitter
    kmf = KaplanMeierFitter()

    ax = plt.subplot(111)

    s_prop = {}

    # Fit and plot each group separately onto the same axis (ax)
    for name, grouped_df in train.groupby("Stage"):
        kmf.fit(
            grouped_df["Survival Time"],
            event_observed=grouped_df["Survival Event"],
            label=name,
        )
        s_prop[name] = dict(kmf.survival_function_at_times([2*365.0]))

    km_risk = 1 - pd.DataFrame.from_dict(s_prop, orient='index')

    df_km_2yr = test['Stage'].map(dict(km_risk[730.0]))

    cindex = ConcordanceIndex()
    cind = cindex(
        torch.from_numpy(df_km_2yr.to_numpy()), 
        torch.from_numpy(test['Survival Event'].to_numpy(dtype=bool)), 
        torch.from_numpy(test['Survival Time'].to_numpy())
    )

    dt = [('Survival Event', '?'), ('Survival Time', '<f8')]
    df = df.astype({'Survival Event': bool, 'Survival Time': float})

    survival_test = np.zeros(len(test), dtype=dt)
    survival_test['Survival Event'] = test['Survival Event'].to_numpy()
    survival_test['Survival Time'] = test['Survival Time'].to_numpy()

    survival_train = np.zeros(len(train), dtype=dt)
    survival_train['Survival Event'] = train['Survival Event'].to_numpy()
    survival_train['Survival Time'] = train['Survival Time'].to_numpy()

    auc, mean_auc = cumulative_dynamic_auc(survival_test, survival_test, df_km_2yr, [730.], tied_tol=1e-08)

    print(f"Split {split}, C Ind: {cind}, AUC: {auc}")

In [ ]:
sample = next(iter(rw_loader))
sample.keys()

dict_keys(['label', 'label_mask', 'path_lang_mask', 'path_lang', 'rad_lang_mask', 'rad_lang', 'clinical_mask', 'clinical', 'slide_ids', 'survival_days', 'survival_right_censor', 'stage'])

In [34]:
#for each sample in loader, need to pull the id and survival time
rih_ids, rih_surv   = [], []
rw_ids, rw_surv     = [], []

for sample in rih_loader:
    rih_ids.append(sample['slide_ids'].item())
    rih_surv.append(sample['survival_days'].item())

for sample in rw_loader:
    rw_ids.append(sample['slide_ids'].item())
    rw_surv.append(sample['survival_days'].item())

rih_df = rih_df[rih_df['UniID of PDAC'].isin(rih_ids)]
rw_df = rw_df[rw_df['Patient ID'].isin(rw_ids)]

In [37]:
def desc_cat_column(df, col, mapping=None, mapping_name = None):
    """
    
    df: pandas dataframe
    col: column in df to summarize
    mapping: map the unique values in col (dict)
    mapping_name: name of mapped column
    """
    desc = pd.concat([df[col].value_counts(), df[col].value_counts(normalize=True)], axis=1)
    if mapping is not None:
        desc[mapping_name] = desc.index.to_series().map(mapping)
        desc = desc[[mapping_name, 'count', 'proportion']]

    return desc

#waht are the covariates relevant here?
#i guess we can do like age, gender, survival time, tumor grade 

#survival time: just grab from dataloader

#RIH DATA: "PDAC IHC.xlsx"
#age: 'Age at Diagnosis'
#gender: 'Sex'
#tumor grade: 'AJCC Pathologic Stage Group'

#RW DATA: Combined_Clinical_lab.xlsx -> make sure to re-export this
#age: 'Age'
#gender: '''Sex \nFemale=0\nMale=1'''
#tumor grade: 'AJCC Pathologic Stage Group'

### Age

In [38]:
print(rih_df['Age at Diagnosis'].describe())
print(rw_df['Age'].describe())

count    70.000000
mean     67.471429
std      11.704974
min      30.000000
25%      60.250000
50%      68.500000
75%      77.000000
max      88.000000
Name: Age at Diagnosis, dtype: float64
count    32.000000
mean     70.062500
std      11.401577
min      29.000000
25%      64.000000
50%      71.500000
75%      75.500000
max      94.000000
Name: Age, dtype: float64


In [39]:
print(desc_cat_column(rih_df, 'Sex'))
print(desc_cat_column(rw_df, 'Sex \nFemale=0\nMale=1', {0: "Female", 1: "Male"}, 'Sex'))

        count  proportion
Sex                      
Male       40    0.571429
Female     30    0.428571
                           Sex  count  proportion
Sex \nFemale=0\nMale=1                           
1.0                       Male     19    0.527778
0.0                     Female     17    0.472222


In [47]:
rih_stage = desc_cat_column(rih_df, 'AJCC Pathologic Stage Group')
rih_stage.index = rih_stage.index.astype(str)
rih_stage = rih_stage.sort_index()
rw_stage = desc_cat_column(rw_df, 'AJCC Pathologic Stage Group')
rw_stage.index = rw_stage.index.astype(str)
rw_stage = rw_stage.sort_index()

print(rih_stage)
print(rw_stage)


                             count  proportion
AJCC Pathologic Stage Group                   
1                                1    0.014286
1A                               3    0.042857
1B                              12    0.171429
2A                               8    0.114286
2B                              30    0.428571
3                                9    0.128571
3A                               2    0.028571
3B                               3    0.042857
4                                2    0.028571
                             count  proportion
AJCC Pathologic Stage Group                   
1B                               4    0.121212
2A                               4    0.121212
2B                              19    0.575758
3                                3    0.090909
3A                               3    0.090909


In [50]:
print(list(rih_ids))
print(len(rih_ids))

[7, 10, 15, 19, 34, 38, 41, 60, 62, 69, 70, 72, 73, 76, 77, 80, 87, 88, 90, 92, 105, 108, 109, 119, 122, 125, 128, 130, 132, 134, 139, 143, 146, 150, 151, 153, 154, 155, 156, 157, 159, 164, 165, 166, 167, 168, 169, 171, 177, 178, 180, 183, 184, 185, 187, 195, 198, 202, 203, 204, 205, 209, 210, 212, 213, 214, 215, 227, 229, 230]
70


In [51]:
print(list(rw_ids))
print(len(rw_ids))

[4, 5, 9, 12, 19, 20, 26, 27, 31, 33, 36, 37, 39, 47, 60, 64, 73, 78, 82, 94, 103, 105, 106, 121, 127, 128, 132, 146, 149, 157, 162, 163, 172, 185, 191, 207]
36


In [ ]:
rih_surv = pd.DataFrame({"Survival (days)": rih_surv})
rw_surv = pd.DataFrame({"Survival (days)": rw_surv})

print(rih_surv["Survival (days)"].describe())
print(rih_surv["Survival (days)"].describe())